In [43]:
import pandas as pd
import numpy as np
import networkx as nx
import time
from algorithms import spnsa, _single_source_shortest_path_basic, calculate_reliance_from_sources, draw_graph
import random

In [44]:
import importlib
import algorithms              # make sure the module itself is imported
importlib.reload(algorithms)   # reloads algorithms.py from disk

from algorithms import spnsa, _single_source_shortest_path_basic, calculate_reliance_from_sources, draw_graph

In [45]:


def create_criminal_graph(file_path:str):
    #G = nx.MultiDiGraph()
    G = nx.DiGraph()

    with open(file_path) as f:
        lines = f.readlines()
        for line in lines[1:]:
            p = [x.strip() for x in line.split(',')]
            if p[10] == '1':
                v = p[2]
                w = p[4]
                G.add_edge(v, w)

    f.close()
    return G

def create_graph(file_path:str):

    #G = nx.MultiDiGraph()
    G = nx.DiGraph()

    with open(file_path) as f:
        lines = f.readlines()
        for line in lines[1:]:
            p = [x.strip() for x in line.split(',')]
            v = p[2]
            w = p[4]
            G.add_edge(v, w)

    f.close()
    return G

def get_suspicious_nodes(file_path:str):
    suspicious_nodes = set()

    with open(file_path) as f:
        lines = f.readlines()
        for line in lines[1:]:
            p = [x.strip() for x in line.split(',')]
            if p[10] == '1':
                suspicious_nodes.add(p[2])
                suspicious_nodes.add(p[4])

    f.close()
    return suspicious_nodes

def get_suspicious_edges(file_path:str):
    suspicious_edges = set()

    with open(file_path) as f:
        lines = f.readlines()
        for line in lines[1:]:
            p = [x.strip() for x in line.split(',')]
            if p[10] == '1':
                suspicious_edges.add((p[2], p[4]))

    f.close()
    return suspicious_edges

def find_largest_component(
    G: nx.Graph,
    component_type: str = "weak",  # used only if G is directed
) -> nx.Graph:
    """
    Return the largest component as an induced subgraph (copied).

    - If G is undirected: uses connected_components.
    - If G is directed: uses weakly_connected_components by default,
      or strongly_connected_components if component_type="strong".
    """
    if G.number_of_nodes() == 0:
        # preserve graph type
        return G.copy()

    if G.is_directed():
        if component_type not in {"weak", "strong"}:
            raise ValueError("component_type must be 'weak' or 'strong' for directed graphs.")
        comps = (
            nx.weakly_connected_components(G)
            if component_type == "weak"
            else nx.strongly_connected_components(G)
        )
    else:
        comps = nx.connected_components(G)

    largest_cc = max(comps, key=len)
    return G.subgraph(largest_cc).copy()

def get_connected_components(G):
    if nx.number_connected_components(G) == 1:
        return [G]
        
    connected_components_generatorset = nx.connected_components(G)
    connected_components = []
    for component in connected_components_generatorset:
        component_graph = G.subgraph(component).copy()
        connected_components.append(component_graph)
    return connected_components

def get_eccentricity_distribution_list(component_list):
    ''' component_list : List[component] '''
    eccentricity_distribution_list = []
    for component in component_list:
        eccentricity_distribution = nx.eccentricity(component)
        eccentricity_distribution_list.append(eccentricity_distribution)
    return eccentricity_distribution_list

In [46]:
G_all = create_graph('../../datasets/IBM AML/LI-Small_Trans.csv')
print(G_all)

DiGraph with 705903 nodes and 1384862 edges


In [47]:
G_criminal = create_criminal_graph('../../datasets/IBM AML/LI-Small_Trans.csv')
print(G_criminal)

DiGraph with 5304 nodes and 3516 edges


In [48]:
# largest component H
largest_component = find_largest_component(G_all)
print(largest_component)

DiGraph with 504391 nodes and 1184998 edges


In [49]:
criminal_nodes = get_suspicious_nodes('../../datasets/IBM AML/LI-Small_Trans.csv')
criminal_nodes_list = list(criminal_nodes)
all_nodes_list = list(largest_component.nodes())
criminal_edges = get_suspicious_edges('../../datasets/IBM AML/LI-Small_Trans.csv')

In [50]:
print(len(criminal_nodes))
print(len(criminal_edges))

5304
3516


In [51]:
print(len(criminal_nodes & largest_component.nodes()))

5285


In [9]:
upto = 10
out_degree = dict(largest_component.out_degree())
out_degree_largest = sorted(out_degree.items(), key=lambda x: x[1], reverse=True)[:upto]
out_degree_largest = dict(out_degree_largest)
out_degree_largest

{'10042B660': 18942,
 '10042B6A8': 11877,
 '10042B6F0': 3611,
 '10042B780': 2620,
 '10042BA51': 2474,
 '10042BA08': 1928,
 '10042B8A0': 1637,
 '10042B930': 1594,
 '10042B738': 1560,
 '10042B9C0': 1370}

In [10]:
upto = 10
in_degree = dict(largest_component.in_degree())
in_degree_largest = sorted(in_degree.items(), key=lambda x: x[1], reverse=True)[:upto]
in_degree_largest = dict(in_degree_largest)
in_degree_largest

{'10042B660': 785,
 '10042B6A8': 478,
 '10042B6F0': 176,
 '10042B780': 118,
 '808283270': 90,
 '80AC3D4E0': 90,
 '818923590': 89,
 '80AEBC5F0': 88,
 '805ADEA90': 87,
 '8187275B0': 87}

In [11]:
upto = 10

degree_diff = {
    node: abs(in_degree.get(node, 0) - out_degree.get(node, 0))
    for node in largest_component.nodes()
}

degree_diff_largest = sorted(degree_diff.items(), key=lambda x: x[1], reverse=True)[:upto]
degree_diff_largest = dict(degree_diff_largest)
degree_diff_largest

{'10042B660': 18157,
 '10042B6A8': 11399,
 '10042B6F0': 3435,
 '10042B780': 2502,
 '10042BA51': 2432,
 '10042BA08': 1842,
 '10042B8A0': 1556,
 '10042B930': 1527,
 '10042B738': 1490,
 '10042B9C0': 1321}

In [12]:
len(degree_diff_largest.keys() & in_degree_largest.keys())

4

# Ex0: Uniformly random feed experiments

In [13]:
# feed size 200; radius 1
number_of_suspicious_nodes = []
number_of_nodes = []
number_of_edges = []
for i in range(100):
    feed = random.sample(all_nodes_list, 200)
    SPN, paths = spnsa(largest_component, feed, radius=1)
    num_suspicious_nodes = len(SPN.nodes() & criminal_nodes)
    number_of_suspicious_nodes.append(num_suspicious_nodes)
    number_of_nodes.append(SPN.number_of_nodes())
    number_of_edges.append(SPN.number_of_edges())
    
avg_num_nodes = sum(number_of_nodes) / len(number_of_nodes)
avg_num_edges = sum(number_of_edges) / len(number_of_edges)
avg_num_suspicious_nodes = sum(number_of_suspicious_nodes) / len(number_of_suspicious_nodes)

print(f'DiGraph with average {avg_num_nodes} and {avg_num_edges} edges')
print(f'Number of suspicious nodes in SPN: {avg_num_suspicious_nodes}')
print(f'Criminal ration in SPN: {avg_num_suspicious_nodes / avg_num_nodes}')

DiGraph with average 367.42 and 167.86 edges
Number of suspicious nodes in SPN: 6.4
Criminal ration in SPN: 0.017418757824832615


In [14]:
# feed size 200; radius 2
number_of_suspicious_nodes = []
number_of_nodes = []
number_of_edges = []

for _ in range(10):
    feed = random.sample(all_nodes_list, 200)
    SPN, paths = spnsa(largest_component, feed, radius=2)
    num_suspicious_nodes = len(SPN.nodes() & criminal_nodes)
    number_of_suspicious_nodes.append(num_suspicious_nodes)
    number_of_nodes.append(SPN.number_of_nodes())
    number_of_edges.append(SPN.number_of_edges())
    
avg_num_nodes = sum(number_of_nodes) / len(number_of_nodes)
avg_num_edges = sum(number_of_edges) / len(number_of_edges)
avg_num_suspicious_nodes = sum(number_of_suspicious_nodes) / len(number_of_suspicious_nodes)

print(f'DiGraph with average {avg_num_nodes} and {avg_num_edges} edges')
print(f'Number of suspicious nodes in SPN: {avg_num_suspicious_nodes}')
print(f'Criminal ration in SPN: {avg_num_suspicious_nodes / avg_num_nodes}')

DiGraph with average 454.4 and 257.1 edges
Number of suspicious nodes in SPN: 7.8
Criminal ration in SPN: 0.01716549295774648


In [15]:
# feed size 500; radius 1
number_of_suspicious_nodes = []
number_of_nodes = []
number_of_edges = []

for _ in range(100):
    feed = random.sample(all_nodes_list, 500)
    SPN, paths = spnsa(largest_component, feed, radius=1)
    num_suspicious_nodes = len(SPN.nodes() & criminal_nodes)
    number_of_suspicious_nodes.append(num_suspicious_nodes)
    number_of_nodes.append(SPN.number_of_nodes())
    number_of_edges.append(SPN.number_of_edges())
    
avg_num_nodes = sum(number_of_nodes) / len(number_of_nodes)
avg_num_edges = sum(number_of_edges) / len(number_of_edges)
avg_num_suspicious_nodes = sum(number_of_suspicious_nodes) / len(number_of_suspicious_nodes)

print(f'DiGraph with average {avg_num_nodes} and {avg_num_edges} edges')
print(f'Number of suspicious nodes in SPN: {avg_num_suspicious_nodes}')
print(f'Criminal ration in SPN: {avg_num_suspicious_nodes / avg_num_nodes}')

DiGraph with average 918.25 and 420.31 edges
Number of suspicious nodes in SPN: 16.0
Criminal ration in SPN: 0.0174244486795535


# Ex1: Largest degree difference feed experiments

In [16]:
# feed size 200; radius 1
feed = dict(sorted(degree_diff.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 246 nodes and 134 edges
Number of suspicious nodes in SPN: 28
Criminal ration in SPN: 0.11382113821138211


In [17]:
# feed size 200; radius 2
feed = dict(sorted(degree_diff.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=2)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 309 nodes and 235 edges
Number of suspicious nodes in SPN: 30
Criminal ration in SPN: 0.0970873786407767


In [18]:
# feed size 500; radius 1
feed = dict(sorted(degree_diff.items(), key=lambda x: x[1], reverse=True)[:500])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 580 nodes and 329 edges
Number of suspicious nodes in SPN: 50
Criminal ration in SPN: 0.08620689655172414


# Ex2: Non-zero in-degree & zero out-degree experiments (Collect)

In [19]:
upto = 10

nonzero_in_degree = {
    node: in_degree.get(node, 0)
    for node in largest_component.nodes() if out_degree.get(node, 0) == 0
}

nonzero_in_degree_largest = sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:upto]
nonzero_in_degree_largest = dict(nonzero_in_degree_largest)
nonzero_in_degree_largest

{'80AC3D4E0': 90,
 '815745460': 86,
 '812588260': 81,
 '8119ACF60': 76,
 '815313820': 74,
 '818649B90': 69,
 '805624EC0': 67,
 '80468F580': 66,
 '818BD7270': 63,
 '804F55B00': 63}

In [20]:
# feed size 200; radius 1
feed = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 200 nodes and 0 edges
Number of suspicious nodes in SPN: 15
Criminal ration in SPN: 0.075


In [21]:
# feed size 200; radius 2
feed = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=2)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 200 nodes and 0 edges
Number of suspicious nodes in SPN: 15
Criminal ration in SPN: 0.075


In [22]:
# feed size 500; radius 1
feed = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:500])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 500 nodes and 0 edges
Number of suspicious nodes in SPN: 29
Criminal ration in SPN: 0.058


# Ex3: Non-zero out-degree & zero in-degree experiments (Distribute)

In [23]:
upto = 10

nonzero_out_degree = {
    node: out_degree.get(node, 0)
    for node in largest_component.nodes() if in_degree.get(node, 0) == 0
}

nonzero_out_degree_largest = sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:upto]
nonzero_out_degree_largest = dict(nonzero_out_degree_largest)
nonzero_out_degree_largest

{'80C88AD30': 26,
 '800140070': 23,
 '800B9DBD0': 20,
 '80037F660': 19,
 '800B106C0': 18,
 '8022F9F20': 18,
 '80214C870': 17,
 '8016594A0': 17,
 '8153D0FF0': 17,
 '80453CC40': 16}

In [24]:
# feed size 200; radius 1
feed = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 485 nodes and 293 edges
Number of suspicious nodes in SPN: 4
Criminal ration in SPN: 0.008247422680412371


In [25]:
# feed size 200; radius 2
feed = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=2)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 706 nodes and 515 edges
Number of suspicious nodes in SPN: 8
Criminal ration in SPN: 0.0113314447592068


In [26]:
# feed size 500; radius 1
feed = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:500])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 1174 nodes and 719 edges
Number of suspicious nodes in SPN: 23
Criminal ration in SPN: 0.019591141396933562


# Ex4: 50% Collect and 50% Distribute feed experiments

In [27]:
# feed size 200; radius 1
upto = 200
nonzero_in_degree_largest = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
nonzero_out_degree_largest = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
feed = nonzero_in_degree_largest | nonzero_out_degree_largest

SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')


DiGraph with 348 nodes and 148 edges
Number of suspicious nodes in SPN: 8
Criminal ration in SPN: 0.022988505747126436


In [28]:
# feed size 200; radius 2
upto = 200
nonzero_in_degree_largest = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
nonzero_out_degree_largest = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
feed = nonzero_in_degree_largest | nonzero_out_degree_largest

SPN, paths = spnsa(largest_component, feed, radius=2)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')


DiGraph with 476 nodes and 276 edges
Number of suspicious nodes in SPN: 11
Criminal ration in SPN: 0.023109243697478993


In [29]:
# feed size 500; radius 1
upto = 500
nonzero_in_degree_largest = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
nonzero_out_degree_largest = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
feed = nonzero_in_degree_largest | nonzero_out_degree_largest

SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')


DiGraph with 856 nodes and 367 edges
Number of suspicious nodes in SPN: 24
Criminal ration in SPN: 0.028037383177570093


# Ex5: High illicit degrees as feed experiments

In [30]:
upto = 10
illicit_degree = dict(G_criminal.degree())
illicit_degree_largest = sorted(illicit_degree.items(), key=lambda x: x[1], reverse=True)[:upto]
illicit_degree_largest = dict(illicit_degree_largest)
illicit_degree_largest

{'10042B660': 308,
 '10042B6A8': 196,
 '10042B6F0': 48,
 '10042BA08': 37,
 '10042B780': 32,
 '10042BA51': 31,
 '10042B9C0': 28,
 '10042B738': 28,
 '10042B858': 25,
 '80219F420': 25}

In [31]:
# feed size 200; radius 1
feed = dict(sorted(illicit_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 449 nodes and 397 edges
Number of suspicious nodes in SPN: 261
Criminal ration in SPN: 0.5812917594654788


In [32]:
# feed size 200; radius 2
feed = dict(sorted(illicit_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=2)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 612 nodes and 600 edges
Number of suspicious nodes in SPN: 313
Criminal ration in SPN: 0.511437908496732


In [33]:
# feed size 500; radius 1
feed = dict(sorted(illicit_degree.items(), key=lambda x: x[1], reverse=True)[:500])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 1027 nodes and 996 edges
Number of suspicious nodes in SPN: 594
Criminal ration in SPN: 0.578383641674781


In [34]:
# random 200 feed; radius 1
import random

feed = dict(random.sample(list(illicit_degree.items()), 200))
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 371 nodes and 179 edges
Number of suspicious nodes in SPN: 257
Criminal ration in SPN: 0.692722371967655
